# Multi-Horizon Quantile Temporal Forecaster (TFT)
## Proposal P3 — Cold Chain EWS Digital Twin

**Student:** Krishna Sikheriya (`IIT2023139`)  
**Supervisor:** Netranand Pathak | **Instructor:** Dr. Shashi Kant Rai  
**Repository:** [coldchain-ews-twin](https://github.com/Krishna200608/coldchain-ews-twin)  

### Novelty Claim:
Replaces reactive anomaly detection with pro-active multi-horizon quantile forecasting (q=0.1, 0.5, 0.9) over a 10-step lookahead horizon using Pinball loss and temporal self-attention. Pre-emptively fires early warning when forecasted trajectory envelope breaches temperature safety limits before the excursion physically occurs.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

### 1. Model Architecture & Pinball Quantile Loss
Encoder-Decoder with temporal GRU, multi-head self-attention, and multi-quantile projection.

In [ ]:
class TemporalQuantileForecaster(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=64, horizon=10, quantiles=None, num_layers=2):
        super().__init__()
        self.horizon = horizon
        self.quantiles = quantiles or [0.1, 0.5, 0.9]
        self.encoder = nn.GRU(input_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.attention = nn.MultiheadAttention(hidden_dim, num_heads=4, batch_first=True)
        self.fc_context = nn.Linear(hidden_dim, hidden_dim)
        self.fc_horizon = nn.Linear(hidden_dim, horizon * hidden_dim // 2)
        self.fc_head = nn.Sequential(nn.ReLU(), nn.Linear(hidden_dim // 2, len(self.quantiles)))

    def forward(self, x):
        B, L, _ = x.shape
        gru_out, _ = self.encoder(x)
        attn_out, _ = self.attention(gru_out, gru_out, gru_out)
        context = F.relu(self.fc_context(attn_out[:, -1, :]))
        h_exp = self.fc_horizon(context).view(B, self.horizon, -1)
        return self.fc_head(h_exp)

def pinball_loss(y_true, y_pred, quantiles=[0.1, 0.5, 0.9]):
    total_loss = 0.0
    for i, q in enumerate(quantiles):
        err = y_true - y_pred[:, :, i]
        total_loss += torch.max((q - 1.0) * err, q * err).mean()
    return total_loss / len(quantiles)

### 2. Load Pretrained Weights & Run Quantile Forecast
Loads checkpoint from models/tft_forecaster_out.pt and plots 10-step quantile trajectory.

In [ ]:
import pathlib
REPO_ROOT = pathlib.Path("..")
model = TemporalQuantileForecaster().to(device)
ckpt = torch.load(REPO_ROOT / "models/tft_forecaster_out.pt", map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print("Model loaded successfully!")

dummy_input = torch.randn(1, 30, 4, device=device)
with torch.no_grad():
    preds = model(dummy_input)[0].cpu().numpy()
print("Forecast shape (10 steps x 3 quantiles):", preds.shape)
print("q=0.1 (lower):", preds[:, 0])
print("q=0.5 (median):", preds[:, 1])
print("q=0.9 (upper):", preds[:, 2])